# **PARTE B: ENGINEERING PROMPTING**

> Para el desarrollo y la optimización de este Notebook, se ha integrado la capacidad analítica de Gemini 3.1 Pro (la versión avanzada y actual de la familia Gemini) junto con el Modo IA de Google, herramientas que han permitido refinar la lógica del código, asegurar la precisión en el procesamiento de lenguaje natural y validar la coherencia semántica de las respuestas generadas.

**¿Qué significa temperature \= 0.5?**

El parámetro de **temperatura** controla la aleatoriedad y la creatividad en las respuestas del LLM. Funciona en una escala (generalmente de 0.0 a 1.0 o 2.0).

* Una temperatura de **0.0** hace que el modelo sea completamente determinista (siempre dará la misma respuesta más probable, ideal para tareas analíticas o matemáticas).  
* Una temperatura de **1.0** hace que el modelo sea muy creativo y variado (ideal para escribir historias o poesía, pero con más riesgo de alucinaciones).  
* **temperature \= 0.5** es un punto intermedio perfecto. Le estamos pidiendo a Gemini que sea lo suficientemente creativo para darnos recomendaciones de películas variadas e interesantes, pero lo suficientemente conservador para no inventarse títulos o salirse del contexto.

---

### 1. Para hacer prompting con Gemini, primero copiamos API Key

Para poder ejecutar el archivo y hacer prompting con Gemini, vais a tener que crearos primero un proyecto y luego una key asociada a ese proyecto en AI studio de Google. Esto se hace aquí: https://aistudio.google.com/

Una vez generada, la copiamos en el apartado `Secretos` 🔑 de Google Colab con el formato: `GOOGLE_API_KEY="yD1D30fEB-7eHPBpsIDg0OXi_s5Wcthbpo"`.

Ahora instalamos las siguientes librerias para utilizar `LangChain`:


In [ ]:
!pip install langchain langchain-google-genai python-dotenv langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.4 MB/s eta 0:00:00


### 2. Preparación del Entorno en el Notebook

El entorno de trabajo se llevará a cabo en Google Colab. Cómo se ha mencionado en el punto anterior. En una celda de este notebook, necesitas cargar la API Key y las librerias como se muestra a continuación.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from google.colab import userdata

Cargamos el API Key desde Googel Colab

In [ ]:
# Sacamos la llave de la caja fuerte de Colab
mi_llave = userdata.get('GOOGLE_API_KEY')

# Se la damos al sistema para que Gemini la pueda usar
os.environ["GOOGLE_API_KEY"] = mi_llave

print("¡Llave cargada correctamente!")

¡Llave cargada correctamente!


O para ejecutar en local, copiamos y pegamos en el archivo local .env con el formato: `GOOGLE_API_KEY="yD1D30fEB-7eHPBpsIDg0OXi_s5Wcthbpo"`. Y luego ejecutamos la siguiente función.

In [ ]:
load_dotenv() # to load the .env file

# Comprobamos que se ha cargado bien leyendo la llave
mi_llave = os.getenv("GOOGLE_API_KEY")

if mi_llave:
    print("¡Éxito! La llave se ha cargado correctamente.")
    # los primeros 5 caracteres por seguridad
    print(f"La llave empieza por: {mi_llave[:5]}...")
else:
    print("Error: No se encontró la llave. Revisa el archivo .env")

### 3. Implementación de los 3 Prompts Libres

A continuación, crearemos las plantillas (Templates) para los tres tipos de prompts solicitados.

**A. Zero-shot prompting**

Dándole a Gemini nuestro rol, contexto, pregunta y formato de output, pídele que te muestre esas recomendaciones.

* Solo se hace la pregunta
* Peor método: no reduce alucinaciones

In [ ]:
plantilla_sin_ejemplos = """
Rol: Eres un experto de cine con un exquisito gusto por la Ciencia Ficción.
Contexto: Estoy buscando opciones de entretenimiento audiovisual que se adapten a mi entorno y estado de ánimo actual.
Pregunta: Recomiéndame 3 películas ideales para ver en un día lluvioso y 3 películas para ver cuando estoy muy feliz.
Formato de output: Una lista con viñetas agrupada por las dos categorías, incluyendo solo el título y el año.

Respuesta:
"""

**B. Few-shot prompting**

De nuevo, Dándole a Gemini nuestro rol, contexto,
pregunta y formato de output, pídele que te muestre esas
recomendaciones. Sin embargo, en este caso le tenéis que dar un ejemplo  de película para tiempo lluvioso, y otra película de ejemplo para cuando se está feliz.

* En el prompt se incluyen ejemplos

In [ ]:
plantilla_con_ejemplos = """
Rol: Eres un experto de cine con un exquisito gusto por la Ciencia Ficción.
Contexto: Estoy buscando opciones de entretenimiento audiovisual que se adapten a mi entorno y estado de ánimo actual.
Pregunta: Recomiéndame 3 películas para un día lluvioso y 3 para cuando estoy feliz.
Formato de output: Una lista con viñetas.

Ejemplos:
- Película para tiempo lluvioso: 'Battlefield Earth'
- Película para cuando estás feliz: 'War of the Worlds'

Sigue este estilo para darme tus 6 recomendaciones nuevas:
"""

**C. Few-shot prompting + Chain-of-Thoughts (CoT)**

Vamos a repetir la petición anterior, pero esta vez dándole un ejemplo de cada película, dándole las razones que os llevan a pensar que esa película de ejemplo es para tiempo lluvioso o película feliz, respectivamente. Podéis incluir un contraejemplo, añadiendo sus razones, también.

* Se pide el razonamiento paso a paso

In [ ]:
plantilla_paso_a_paso = """
Rol: Eres un crítico de cine experto.
Contexto: Necesito recomendaciones adaptadas al clima y mi humor.
Pregunta: Recomiéndame 3 películas para un día lluvioso y 3 para cuando estoy feliz. Explica paso a paso por qué.
Formato de output: Lista con título y explicación paso a paso del porqué.

Ejemplos con razonamiento:
- Lluvia: 'Se7en'. Razón: Un día lluvioso pide misterio; la película transcurre casi enteramente lloviendo, complementando la atmósfera de estar a cubierto.
- Feliz: 'La La Land'. Razón: Cuando estás feliz buscas colores vibrantes y música estimulante que eleve tu energía.
- Contraejemplo: No recomendaría 'La Lista de Schindler' para un día feliz. Razón: Su tono histórico y trágico choca con un estado de ánimo alegre y lo arruinaría.

Siguiendo esta misma lógica de razonamiento paso a paso, dame tus recomendaciones:
"""

**D. Acotado a los resultados del recomendador**

Repite los 3 prompts anteriores, pero acotando las soluciones a las
recomendaciones que nos ha dado el recomendador en la parte A para el
usuario que habéis escogido aleatoriamente del punto 8.

Para que funcione, tenemos que crear nuevas plantillas (o añadir el texto a las que tenemos) que incluyan explícitamente la restricción y la variable {lista_recomendador}.

In [ ]:
mis_peliculas_recomendadas = """
 1. Wrong Trousers,
 2. One Flew Over the Cuckoo's Nest,
 3. Shawshank Redemption,
 4. Rear Window,
 5. North by Northwest,
 6. Mr. Smith Goes to Washington,
 7. Usual Suspects,
 8. Wallace & Gromit: The Best of Aardman Animation,
 9. Hoop Dreams,
 10. Godfather
"""

# 1. Creamos las nuevas plantillas con la RESTRICCIÓN y la variable {lista_recomendador}
plantilla_zero_acotado = plantilla_sin_ejemplos + """
RESTRICCIÓN OBLIGATORIA: Tienes que elegir tus 6 recomendaciones EXCLUSIVAMENTE de esta lista:
{lista_recomendador}
"""

plantilla_few_acotado = plantilla_con_ejemplos + """
RESTRICCIÓN OBLIGATORIA: Tienes que elegir tus 6 recomendaciones EXCLUSIVAMENTE de esta lista:
{lista_recomendador}
"""

plantilla_cot_acotado = plantilla_paso_a_paso + """
RESTRICCIÓN OBLIGATORIA: Tienes que elegir tus 6 recomendaciones EXCLUSIVAMENTE de esta lista:
{lista_recomendador}.
Si es difícil encajar el clima/humor con las opciones, razona por qué elegiste esa opción sobre las demás de tu lista permitida.
"""

  Función para ejecutar la práctica de los 3 tipos de prompts + acotación usando el modelo Gemini configurado.

In [ ]:
def practica_prompts():
    modelo = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GOOGLE_API_KEY"), temperature=0.5)

    print("\\n" + "="*40 + " A. ZERO-SHOT PROMPTING " + "="*40)
    prompt_zero = ChatPromptTemplate.from_template(plantilla_sin_ejemplos)
    cadena_zero = prompt_zero | modelo
    print("--- RESULTADOS ZERO-SHOT ---")
    print(cadena_zero.invoke({}).content)

    print("\\n" + "="*40 + " B. FEW-SHOT PROMPTING " + "="*40)
    prompt_few = ChatPromptTemplate.from_template(plantilla_con_ejemplos)
    cadena_few = prompt_few | modelo
    print("\n--- RESULTADOS FEW-SHOT ---")
    print(cadena_few.invoke({}).content)

    print("\\n" + "="*40 + " C. FEW-SHOT + CHAIN-OF-THOUGHTS (CoT) " + "="*40)
    prompt_cot = ChatPromptTemplate.from_template(plantilla_paso_a_paso)
    cadena_cot = prompt_cot | modelo
    print("\n--- RESULTADOS FEW-SHOT + CoT ---")
    print(cadena_cot.invoke({}).content)

    #"""
    # D. ACOTADO A LOS RESULTADOS DEL RECOMENDADOR
    print("\\n" + "="*40 + " D. ACOTADO AL RECOMENDADOR " + "="*40)
    prompt_zero_acotado = ChatPromptTemplate.from_template(plantilla_zero_acotado)
    cadena_zero_acotada = prompt_zero_acotado | modelo
    print("--- RESULTADOS ZERO-SHOT ---")
    print(cadena_zero_acotada.invoke({"lista_recomendador": mis_peliculas_recomendadas}).content)

    prompt_few_acotado = ChatPromptTemplate.from_template(plantilla_few_acotado)
    cadena_few_acotada = prompt_few_acotado | modelo
    print("\n--- RESULTADOS FEW-SHOT ---")
    print(cadena_few_acotada.invoke({"lista_recomendador": mis_peliculas_recomendadas}).content)

    prompt_cot_acotado = ChatPromptTemplate.from_template(plantilla_cot_acotado)
    cadena_cot_acotada = prompt_cot_acotado | modelo
    print("\n--- RESULTADOS FEW-SHOT + CoT ---")
    print(cadena_cot_acotada.invoke({"lista_recomendador": mis_peliculas_recomendadas}).content)
    #"""

Para que funcione, tienes que crear nuevas plantillas (o añadir el texto a las que tienes) que incluyan explícitamente la restricción y la variable {lista_recomendador}.

Bueno y vamos con el la función main

In [ ]:
def main():
    # Llamada a la función
    print(practica_prompts.__doc__)
    practica_prompts()

if __name__ == "__main__":
    main()

None
\n======================================== A. ZERO-SHOT PROMPTING ========================================
--- RESULTADOS ZERO-SHOT ---
¡Excelente! Como experto en ciencia ficción, me complace ayudarte a navegar por el vasto universo cinematográfico. Aquí tienes mis selecciones, cuidadosamente elegidas para cada estado de ánimo:

**Para un día lluvioso (pensativas y atmosféricas):**

*   Blade Runner (1982)
*   Arrival (2016)
*   Gattaca (1997)

**Para cuando estás muy feliz (estimulantes y optimistas):**

*   Star Wars: Episode IV - A New Hope (1977)
*   Back to the Future (1985)
*   Guardians of the Galaxy (2014)
\n======================================== B. FEW-SHOT PROMPTING ========================================

--- RESULTADOS FEW-SHOT ---
¡Ah, un colega buscador de experiencias audiovisuales que resuenen con el alma! Excelente. Como experto en Ciencia Ficción con un paladar refinado, he meditado cuidadosamente tus necesidades. Aquí tienes mis selecciones, diseñadas para e


### 4. Conclusiones (El informe final)

**I\. Análisis de los 3 prompts SIN acotar (Universo libre)**

Cuando el modelo podía elegir cualquier película de la historia del cine:

* **Zero-Shot:** El modelo hace exactamente lo que le pedimos de forma rápida, pero **se deja influenciar enormemente por el "Rol"** que le asignamos. Como le dijimos que era un "experto en ciencia ficción", todas sus recomendaciones (Blade Runner, Star Wars, Arrival) son estrictamente de ese género, aunque quizás no sean las *mejores* películas universales para representar la "felicidad".  
* **Few-Shot:** Mejora en la presentación. El modelo calca exactamente el formato de viñetas que le ordenamos en los ejemplos (Película para tiempo lluvioso: '...'). Sigue anclado al sesgo de la ciencia ficción.  
* **Few-Shot \+ CoT (Chain of Thought):** **La calidad da un salto gigante.** Al obligarle a pensar "paso a paso" (*Paso 1, Paso 2, Paso 3...*), el modelo se da cuenta de que la prioridad es el estado de ánimo y no solo la ciencia ficción. Fijémonos que aquí recomienda *Paddington 2* o *Cantando bajo la lluvia* para la felicidad. Al forzarle a justificar el "por qué", el modelo produce resultados emocionalmente mucho más inteligentes y precisos.

**II\. Análisis de los 3 prompts ACOTADOS (Con la lista del recomendador)**

Aquí es donde entra en juego la restricción de utilizar solo tu lista (Godfather, Rear Window, Wrong Trousers, etc.).

* **Detección del conflicto:** Lo primero que destaca es que el modelo **se da cuenta del choque de instrucciones**. En el Zero-Shot y Few-Shot acotados se queja amablemente: *"Aunque mi pasión arde por las galaxias... entiendo que esta selección no abunda en ciencia ficción"*. Es decir, el LLM es capaz de identificar que nuestra lista impuesta no encaja con su rol, pero obedece la restricción priorizando la lista.  
* **Zero y Few-Shot Acotado:** Logran el objetivo mecánicamente. Agrupan las películas de la lista en las dos categorías basándose en su conocimiento previo (ej. *Wrong Trousers* es comedia, va a feliz; *Godfather* es oscura, va a lluvia).  
* **CoT Acotado (La joya de la práctica):** Aquí vemos el verdadero poder de los LLMs. Como nuestra lista de películas clásicas no tiene opciones "obvias" para la lluvia o la felicidad, el modelo tiene que hacer malabares lógicos (razonamiento profundo) para justificar la elección:  
  * **Creatividad lógica:** Justifica *Rear Window* (La ventana indiscreta) para un día lluvioso no porque llueva en la película, sino porque transmite la sensación de "estar encerrado en casa espiando a los vecinos", lo cual es una actividad de día lluvioso. ¡Es una deducción brillante\!  
  * **Análisis comparativo:** Fijémonos que el modelo incluye un apartado de *"Razonamiento sobre otras opciones"*. El propio modelo nos explica por qué eligió *El Padrino* y descartó *Cadena Perpetua* (porque para un día entero de lluvia en casa viene mejor una película épica y larga).

**III\. Conclusión Final (Respuesta a la pregunta de la práctica)**

**¿Hay algún tipo de prompt que funcione mejor? ¿Por qué?**  
El **Few-Shot \+ Chain-of-Thoughts (CoT)** es sin duda el mejor. Funciona mejor porque al obligar al modelo a verbalizar su proceso de toma de decisiones (explicar por qué), se reduce la superficialidad, se evitan las respuestas automáticas (clichés) y se generan recomendaciones con una conexión emocional real y argumentada.  
**¿Son los resultados igual de buenos si se incluyen los resultados del recomendador o no?**  
Los resultados acotados no son "peores", sino **diferentes y mucho más útiles en un entorno real**.

* *Sin acotar:* El LLM busca la película ideal en el vacío, pero no sabe si al usuario le gusta ese género.  
* *Acotado:* Garantizamos que al usuario le va a gustar la película (porque la lista viene de un recomendador basado en sus gustos previos), y usamos el LLM con CoT simplemente como un "vendedor genial" que sabe explicarle al usuario *por qué* una de sus películas favoritas es perfecta para verla *precisamente hoy* que está lloviendo. Es la sinergia perfecta entre el Machine Learning tradicional (Recomendador) y la IA Generativa (LLM).